# Κεφάλαιο 7: Δημιουργία Εφαρμογών Συνομιλίας
## Γρήγορη Έναρξη με το Microsoft Foundry Models API

Αυτό το σημειωματάριο είναι προσαρμοσμένο από το [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) που περιλαμβάνει σημειωματάρια που έχουν πρόσβαση στις υπηρεσίες [Azure OpenAI](notebook-azure-openai.ipynb).

> **Σημείωση:** Το GitHub Models σταματά στα τέλη Ιουλίου 2026. Αυτό το σημειωματάριο στοχεύει τώρα στα [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), που προσφέρει τον ίδιο κατάλογο μοντέλων με δωρεάν δοκιμή και την εμπειρία του Azure AI Inference SDK.


# Επισκόπηση  
"Τα μεγάλα γλωσσικά μοντέλα είναι συναρτήσεις που αντιστοιχούν κείμενο σε κείμενο. Δίνοντας μια ακολουθία κειμένου ως είσοδο, ένα μεγάλο γλωσσικό μοντέλο προσπαθεί να προβλέψει το κείμενο που θα ακολουθήσει"(1). Αυτό το σημειωματάριο "γρήγορης εκκίνησης" θα εισαγάγει τους χρήστες σε βασικές έννοιες των LLM, τις κύριες απαιτήσεις πακέτων για να ξεκινήσουν με το AML, μια ήπια εισαγωγή στο σχεδιασμό προτροπών και αρκετά σύντομα παραδείγματα διαφορετικών περιπτώσεων χρήσης. 


## Περιεχόμενα  

[Επισκόπηση](#overview)  
[Πώς να χρησιμοποιήσετε την Υπηρεσία OpenAI](#how-to-use-openai-service)  
[1. Δημιουργία της Υπηρεσίας OpenAI σας](#1.-creating-your-openai-service)  
[2. Εγκατάσταση](#2.-installation)    
[3. Διαπιστευτήρια](#3.-credentials)  

[Περιπτώσεις Χρήσης](#use-cases)    
[1. Περίληψη Κειμένου](#1.-summarize-text)  
[2. Ταξινόμηση Κειμένου](#2.-classify-text)  
[3. Δημιουργία Νέων Ονομάτων Προϊόντων](#3.-generate-new-product-names)  
[4. Βελτίωση Ταξινομητή](#4.fine-tune-a-classifier)  

[Αναφορές](#references)


### Δημιουργήστε το πρώτο σας prompt  
Αυτή η σύντομη άσκηση θα παρέχει μια βασική εισαγωγή για την υποβολή prompt σε ένα μοντέλο στο Microsoft Foundry Models για μια απλή εργασία "περιληπτική παρουσίαση".


**Βήματα**:  
1. Εγκαταστήστε τη βιβλιοθήκη `azure-ai-inference` στο περιβάλλον python σας, εάν δεν το έχετε κάνει ήδη.  
2. Φορτώστε τις τυπικές βοηθητικές βιβλιοθήκες και ρυθμίστε τα διαπιστευτήρια σας για το Microsoft Foundry Models.  
3. Επιλέξτε ένα μοντέλο για την εργασία σας  
4. Δημιουργήστε ένα απλό prompt για το μοντέλο  
5. Υποβάλετε το αίτημά σας στο API του μοντέλου!


### 1. Εγκατάσταση `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Εισαγωγή βοηθητικών βιβλιοθηκών και δημιουργία διαπιστευτηρίων


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Εύρεση του σωστού μοντέλου  
Τα μοντέλα όπως τα GPT-4o και GPT-4o mini μπορούν να κατανοήσουν και να παράγουν φυσική γλώσσα, και είναι διαθέσιμα στον κατάλογο Microsoft Foundry Models μαζί με μοντέλα από τις Meta, Mistral, Cohere και Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. Σχεδιασμός Προτροπής  

"Η μαγεία των μεγάλων γλωσσικών μοντέλων είναι ότι μέσω της εκπαίδευσης για ελαχιστοποίηση του σφάλματος πρόβλεψης πάνω σε τεράστιες ποσότητες κειμένου, τα μοντέλα καταλήγουν να μαθαίνουν έννοιες χρήσιμες για αυτές τις προβλέψεις. Για παράδειγμα, μαθαίνουν έννοιες όπως"(1):

* πώς να γράφουν ορθογραφικά
* πώς λειτουργεί η γραμματική
* πώς να παραφράζουν
* πώς να απαντούν σε ερωτήσεις
* πώς να διατηρούν μια συνομιλία
* πώς να γράφουν σε πολλές γλώσσες
* πώς να προγραμματίζουν
* κ.ά.

#### Πώς να ελέγξετε ένα μεγάλο γλωσσικό μοντέλο  
"Από όλα τα εισερχόμενα σε ένα μεγάλο γλωσσικό μοντέλο, το πιο επιδραστικό μακράν είναι η προτροπή κειμένου(1).

Τα μεγάλα γλωσσικά μοντέλα μπορούν να προτρεχθούν να παράγουν έξοδο με μερικούς τρόπους:

Οδηγία: Πείτε στο μοντέλο τι θέλετε
Ολοκλήρωση: Παρακινήστε το μοντέλο να ολοκληρώσει την αρχή αυτού που θέλετε
Επίδειξη: Δείξτε στο μοντέλο τι θέλετε, με είτε:
Μερικά παραδείγματα στην προτροπή
Εκατοντάδες ή χιλιάδες παραδείγματα σε ένα σύνολο εκπαίδευσης για βελτιστοποίηση"



#### Υπάρχουν τρεις βασικές οδηγίες για τη δημιουργία προτροπών:

**Δείξτε και πείτε**. Κάντε σαφές τι θέλετε είτε μέσω οδηγιών, παραδειγμάτων ή συνδυασμού των δύο. Αν θέλετε το μοντέλο να κατατάξει μια λίστα αντικειμένων σε αλφαβητική σειρά ή να ταξινομήσει μια παράγραφο ανά συναίσθημα, δείξτε του ότι αυτό θέλετε.

**Παρέχετε ποιοτικά δεδομένα**. Αν προσπαθείτε να δημιουργήσετε έναν ταξινομητή ή να κάνετε το μοντέλο να ακολουθήσει ένα μοτίβο, βεβαιωθείτε ότι υπάρχουν αρκετά παραδείγματα. Φροντίστε να διορθώσετε τα παραδείγματά σας — το μοντέλο συνήθως είναι αρκετά έξυπνο ώστε να διακρίνει βασικά ορθογραφικά λάθη και να σας δώσει απάντηση, αλλά μπορεί επίσης να υποθέσει πως αυτό είναι σκόπιμο και αυτό να επηρεάσει την απάντηση.

**Ελέγξτε τις ρυθμίσεις σας.** Οι ρυθμίσεις temperature και top_p ελέγχουν πόσο ντετερμινιστικό είναι το μοντέλο στην παραγωγή απάντησης. Αν ζητάτε απάντηση όπου υπάρχει μόνο μια σωστή απάντηση, τότε θέλετε να τις ρυθμίσετε χαμηλότερα. Αν ψάχνετε για πιο ποικίλες απαντήσεις, τότε μπορεί να θέλετε να τις ρυθμίσετε υψηλότερα. Το πιο συνηθισμένο λάθος που κάνουν οι άνθρωποι με αυτές τις ρυθμίσεις είναι ότι νομίζουν πως είναι έλεγχοι "ευφυΐας" ή "δημιουργικότητας".


Πηγή: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Υποβολή!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Επανάλαβε την ίδια κλήση, πώς συγκρίνονται τα αποτελέσματα;


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Περίληψη Κειμένου  
#### Πρόκληση  
Περίληψη κειμένου με την προσθήκη «tl;dr:» στο τέλος ενός κειμενικού αποσπάσματος. Παρατηρήστε πώς το μοντέλο κατανοεί πώς να εκτελεί διάφορες εργασίες χωρίς επιπλέον οδηγίες. Μπορείτε να πειραματιστείτε με πιο περιγραφικές εντολές από το tl;dr για να τροποποιήσετε τη συμπεριφορά του μοντέλου και να προσαρμόσετε την περίληψη που λαμβάνετε(3).  

Πρόσφατη έρευνα έχει δείξει σημαντικές βελτιώσεις σε πολλές εργασίες και σημεία αναφοράς της Επεξεργασίας Φυσικής Γλώσσας (NLP) με προεκπαίδευση σε ένα μεγάλο σώμα κειμένου, ακολουθούμενη από εκμάθηση εξειδικευμένη σε μια συγκεκριμένη εργασία. Αν και συνήθως η αρχιτεκτονική είναι ανεξάρτητη από την εργασία, αυτή η μέθοδος απαιτεί ακόμη σύνολα δεδομένων εκμάθησης εξειδικευμένα για την εργασία, με χιλιάδες ή δεκάδες χιλιάδες παραδείγματα. Αντίθετα, οι άνθρωποι μπορούν γενικά να εκτελέσουν μια νέα γλωσσική εργασία με λίγα μόνο παραδείγματα ή απλές οδηγίες - κάτι που τα τρέχοντα συστήματα NLP δυσκολεύονται ακόμη να κάνουν σε μεγάλο βαθμό. Εδώ δείχνουμε ότι η κλιμάκωση των γλωσσικών μοντέλων βελτιώνει σημαντικά την απόδοση χωρίς εξειδίκευση, με λίγα παραδείγματα, φτάνοντας μερικές φορές να ανταγωνίζεται προηγούμενες τεχνικές εκμάθησης με εξειδίκευση.  



Tl;dr


# Ασκήσεις για διάφορες περιπτώσεις χρήσης  
1. Περίληψη Κειμένου  
2. Ταξινόμηση Κειμένου  
3. Δημιουργία Νέων Ονομάτων Προϊόντων


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Ταξινόμηση Κειμένου  
#### Πρόκληση  
Ταξινομήστε αντικείμενα σε κατηγορίες που παρέχονται κατά το χρόνο της ερμηνείας. Στο παρακάτω παράδειγμα, παρέχουμε τόσο τις κατηγορίες όσο και το κείμενο προς ταξινόμηση στην προτροπή(*playground_reference). 

Ερώτημα Πελάτη: Γεια σας, ένα από τα πλήκτρα στο πληκτρολόγιο του φορητού μου υπολογιστή έσπασε πρόσφατα και θα χρειαστώ αντικατάσταση:

Ταξινομημένη κατηγορία:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Δημιουργία Νέων Ονομάτων Προϊόντων
#### Πρόκληση
Δημιουργήστε ονόματα προϊόντων από παραδείγματα λέξεων. Εδώ συμπεριλαμβάνουμε στην υπόδειξη πληροφορίες σχετικά με το προϊόν για το οποίο πρόκειται να δημιουργήσουμε ονόματα. Παρέχουμε επίσης ένα παρόμοιο παράδειγμα για να δείξουμε το μοτίβο που θέλουμε να λάβουμε. Έχουμε επίσης ορίσει τη θερμοκρασία σε υψηλή τιμή για να αυξήσουμε την τυχαιότητα και τις πιο καινοτόμες απαντήσεις.

Περιγραφή προϊόντος: Μηχάνημα για milkshake στο σπίτι
Λέξεις εκκίνησης: γρήγορο, υγιεινό, συμπαγές.
Ονόματα προϊόντων: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Περιγραφή προϊόντος: Ένα ζευγάρι παπούτσια που μπορεί να προσαρμοστεί σε κάθε μέγεθος ποδιού.
Λέξεις εκκίνησης: προσαρμόσιμο, εφαρμοστό, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Αναφορές  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Καλύτερες πρακτικές για fine-tuning μοντέλων GPT για ταξινόμηση κειμένου](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Για Περισσότερη Βοήθεια  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Συνεργάτες
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
